# Fresh Retail: Starter Notebook

This notebook accompanies the **Introduction** slide deck (`FreshRetail_Introduction.pptx`). It provides a shared data pipeline for both tracks, then produces the exact outputs previewed in the showcase slides.

**Run sections 1–4 first** (shared setup), then run the section for your track:

| Section | Track | What you produce |
|---------|-------|-----------------|
| 5. Operations | Ops | Temporal profiles, heatmaps, KPIs, hourly patterns |
| 6. Data Science | DS | WAPE baselines, forecast overlays, demand recovery, error analysis |

Both tracks use the same dataset, same helper functions, and same time split.

- **Operations Track**: O1 (Diagnosis) or O2 (Decision)
- **Data Science Track**: D1 (Direct benchmark) or D2 (Recovery first)

**Dataset**: [Dingdong-Inc/FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K)

---
## 1. Setup and Data Download

In [20]:
# Run this cell on Google Colab (already installed locally)
!pip install -q pandas pyarrow matplotlib seaborn datasets

In [21]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Setup complete.")

Setup complete.


In [ ]:
from datasets import load_dataset

print("Downloading FreshRetailNet-50K from Hugging Face...")
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
print(ds)

# Convert to pandas
train_raw = ds["train"].to_pandas()
eval_raw = ds["eval"].to_pandas()

print(f"\nTrain: {train_raw.shape}, Eval: {eval_raw.shape}")
print(f"Columns: {list(train_raw.columns)}")

---
## 2. Data Preparation

In [ ]:
def prepare_panel(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare the raw HF dataset into a clean analysis panel."""
    df = df.copy()

    # Parse date
    df["dt"] = pd.to_datetime(df["dt"])
    df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)

    # Create series_id (unique store x product combination)
    series_keys = df[["store_id", "product_id"]].drop_duplicates().reset_index(drop=True)
    series_keys["series_id"] = range(1, len(series_keys) + 1)
    df = df.merge(series_keys, on=["store_id", "product_id"], how="left")

    # Create day index (days since start)
    min_date = df["dt"].min()
    df["day_idx"] = (df["dt"] - min_date).dt.days + 1

    n_series = df["series_id"].nunique()
    n_days = df["day_idx"].nunique()
    print(f"Prepared {len(df):,} rows \u2014 {n_series:,} series x {n_days} days")
    print(f"Date range: {df['dt'].min().date()} to {df['dt'].max().date()}")
    return df


history = prepare_panel(train_raw)
history.head()

---
## 3. Shared Functions: flag_censoring, make_features, time_split

In [5]:
def flag_censoring(df: pd.DataFrame) -> pd.DataFrame:
    """Add censoring flags based on stockout hours."""
    df = df.copy()
    df["is_censored"] = (df["stock_hour6_22_cnt"] > 0).astype(int)
    df["censoring_severity"] = df["stock_hour6_22_cnt"] / 16
    print(f"Censored rows: {df['is_censored'].sum():,} / {len(df):,} ({df['is_censored'].mean():.1%})")
    return df


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add lag and rolling features for EDA and forecasting."""
    df = df.sort_values(["series_id", "day_idx"]).copy()
    grp = df.groupby("series_id")["sale_amount"]
    df["sales_lag1"] = grp.shift(1)
    df["sales_lag7"] = grp.shift(7)
    df["sales_roll7"] = grp.transform(lambda x: x.rolling(7, min_periods=1).mean())
    df["sales_roll28"] = grp.transform(lambda x: x.rolling(28, min_periods=1).mean())
    df["psd"] = grp.transform("mean")  # per-series daily mean
    return df


def time_split(df: pd.DataFrame, horizon: int = 7) -> tuple:
    """Split into train and validation by time. Validation = last `horizon` days."""
    min_day = df["day_idx"].min()
    max_day = df["day_idx"].max()
    val_start = max_day - horizon + 1
    train = df[df["day_idx"] < val_start].copy()
    val = df[df["day_idx"] >= val_start].copy()
    print(f"Train: day {min_day}..{val_start - 1} ({len(train):,} rows), Val: day {val_start}..{max_day} ({len(val):,} rows)")
    return train, val

In [6]:
# Apply shared pipeline
history = flag_censoring(history)
history = make_features(history)

train, val = time_split(history, horizon=7)
print(f"\nValidation window: day {val['day_idx'].min()} to {val['day_idx'].max()}")

Censored rows: 1,992,006 / 4,500,000 (44.3%)
Train: day 1..83 (4,150,000 rows), Val: day 84..90 (350,000 rows)

Validation window: day 84 to 90


---
## 4. Data at a Glance

In [ ]:
# Show a real series with stockouts
series_stockouts = history.groupby("series_id")["is_censored"].mean()
example_sid = series_stockouts[(series_stockouts > 0.3) & (series_stockouts < 0.7)].index[0]

s_example = history[history["series_id"] == example_sid][
    ["dt", "day_idx", "sale_amount", "stock_hour6_22_cnt", "is_censored", "discount", "holiday_flag", "avg_temperature"]
].head(14)
print(f"Series {example_sid} \u2014 first 14 days (a product with frequent stockouts):")
display(s_example)

In [ ]:
# Dataset dimensions
summary = pd.Series({
    "Total rows": f"{len(history):,}",
    "Series (store x product)": f"{history['series_id'].nunique():,}",
    "Days per series": str(history["day_idx"].nunique()),
    "Products (product_id)": str(history["product_id"].nunique()),
    "Stores (store_id)": str(history["store_id"].nunique()),
    "Cities (city_id)": str(history["city_id"].nunique()),
    "Management groups": str(history["management_group_id"].nunique()),
    "Mean daily sales": f"{history['sale_amount'].mean():.3f}",
    "Censored rows": f"{history['is_censored'].sum():,} ({history['is_censored'].mean():.1%})",
    "Low-sale series (psd<1)": f"{(history.groupby('series_id')['psd'].first() < 1).sum():,}",
    "High-sale series (psd>=1)": f"{(history.groupby('series_id')['psd'].first() >= 1).sum():,}",
})
display(summary.to_frame("Value"))

In [ ]:
# Sales distribution and per-series daily mean
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

clipped = history["sale_amount"].clip(upper=history["sale_amount"].quantile(0.99))
axes[0].hist(clipped, bins=50, color="#065A82", edgecolor="white")
axes[0].set_title("Distribution of daily sales (clipped at 99th pctl)")
axes[0].set_xlabel("sale_amount")
axes[0].set_ylabel("Count")

psd_vals = history.groupby("series_id")["psd"].first()
axes[1].hist(psd_vals, bins=50, color="#1C7293", edgecolor="white")
axes[1].axvline(1.0, color="#E74C3C", linestyle="--", linewidth=2, label="psd=1 cutoff")
axes[1].set_title("Per-series daily mean (psd) distribution")
axes[1].set_xlabel("psd")
axes[1].set_ylabel("Number of series")
axes[1].legend()

plt.tight_layout()
plt.show()

---
---
# DATA SCIENCE TRACK

## 6. Data Science Track

> **You may skip this section if you are focusing on the Operations track.**

This section produces the following outputs — each corresponds to a slide in the deck:

| Output | What it shows | Slide |
|--------|--------------|-------|
| WAPE results table (D1) | Baseline comparison: global mean vs seasonal naive vs rolling 28d | Slide 18 |
| Forecast overlay chart | Predicted vs actual for one series across the validation window | Slide 18 |
| Recovery comparison table (D2) | WAPE on raw vs corrected target — does imputation help? | Slide 19 |
| WAPE by management group | Which product groups are hardest to forecast? | Slide 20 |
| Residual histogram + error scatter | Where the model fails and why | Slide 20 |

**A strong data science project** starts from these baselines and improves on them with better features, better imputation, or a more sophisticated model — always measured by WAPE on the same time split.

### 6a. WAPE Evaluation Function

In [7]:
def compute_wape(actual: np.ndarray, predicted: np.ndarray) -> float:
    """Weighted Absolute Percentage Error."""
    denom = np.sum(np.abs(actual))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(actual - predicted)) / denom


def evaluate_forecast(val_df: pd.DataFrame, pred_col: str = "prediction") -> dict:
    """Compute WAPE overall, low-sale, high-sale, and harmonic mean.
    Only evaluates rows where stock_hour6_22_cnt == 0 (uncensored in validation)."""
    scored = val_df[val_df["stock_hour6_22_cnt"] == 0].copy()
    if len(scored) == 0:
        return {"wape_overall": np.nan}

    y = scored["sale_amount"].values
    yhat = scored[pred_col].values

    wape_all = compute_wape(y, yhat)

    low = scored[scored["psd"] < 1]
    high = scored[scored["psd"] >= 1]

    wape_low = compute_wape(low["sale_amount"].values, low[pred_col].values) if len(low) > 0 else np.nan
    wape_high = compute_wape(high["sale_amount"].values, high[pred_col].values) if len(high) > 0 else np.nan

    if np.isnan(wape_low) or np.isnan(wape_high) or wape_all == 0 or wape_low == 0 or wape_high == 0:
        hm = np.nan
    else:
        hm = 3 / (1/wape_all + 1/wape_low + 1/wape_high)

    return {
        "wape_overall": round(wape_all, 4) if not np.isnan(wape_all) else np.nan,
        "wape_low_sale": round(wape_low, 4) if not np.isnan(wape_low) else np.nan,
        "wape_high_sale": round(wape_high, 4) if not np.isnan(wape_high) else np.nan,
        "harmonic_mean": round(hm, 4) if not np.isnan(hm) else np.nan,
        "scored_rows": len(scored),
    }

print("Evaluation function ready.")

Evaluation function ready.


### 6b. D1 \u2014 Direct Benchmark: Naive Baselines on Raw Sales

In [ ]:
# --- Baseline 1: Global mean ---
series_mean = train.groupby("series_id")["sale_amount"].mean().rename("pred_global_mean")
val = val.drop(columns=["pred_global_mean", "pred_seasonal_naive", "pred_roll28", "forecast_day"], errors="ignore")
val = val.merge(series_mean, on="series_id", how="left")

# --- Baseline 2: Seasonal naive (last-week repeat) ---
val_start = val["day_idx"].min()
last_week = history[history["day_idx"].between(val_start - 7, val_start - 1)][["series_id", "day_idx", "sale_amount"]].copy()
last_week["forecast_day"] = last_week["day_idx"] + 7
last_week = last_week.rename(columns={"sale_amount": "pred_seasonal_naive"})

val = val.merge(last_week[["series_id", "forecast_day", "pred_seasonal_naive"]],
                left_on=["series_id", "day_idx"], right_on=["series_id", "forecast_day"], how="left")
val = val.drop(columns=["forecast_day"], errors="ignore")
val["pred_seasonal_naive"] = val["pred_seasonal_naive"].fillna(val["pred_global_mean"])

# --- Baseline 3: Rolling 28-day mean ---
roll28 = train.groupby("series_id")["sale_amount"].apply(
    lambda x: x.tail(28).mean(), include_groups=False
).rename("pred_roll28")
val = val.merge(roll28, on="series_id", how="left")

# Evaluate all three
results = {}
for method, col in [("Global mean", "pred_global_mean"), ("Seasonal naive", "pred_seasonal_naive"), ("Rolling 28d", "pred_roll28")]:
    val["prediction"] = val[col].clip(lower=0)
    results[method] = evaluate_forecast(val)

results_df = pd.DataFrame(results).T
print("=== D1 Benchmark Results ===")
display(results_df)

In [ ]:
# --- Visualize: forecast overlay for one series ---
example_sid3 = history.groupby("series_id")["psd"].first().sort_values(ascending=False).index[5]
ex = history[history["series_id"] == example_sid3].copy()
ex_val = val[val["series_id"] == example_sid3].copy()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ex["day_idx"], ex["sale_amount"], color="#065A82", linewidth=1.5, label="Actual (train)")
ax.plot(ex_val["day_idx"], ex_val["sale_amount"], color="#065A82", linewidth=2, linestyle="-", label="Actual (val)")
ax.plot(ex_val["day_idx"], ex_val["pred_global_mean"], color="#E67E22", linewidth=1.5, linestyle="--", label="Global mean")
ax.plot(ex_val["day_idx"], ex_val["pred_seasonal_naive"], color="#8E44AD", linewidth=1.5, linestyle="--", label="Seasonal naive")
ax.plot(ex_val["day_idx"], ex_val["pred_roll28"], color="#27AE60", linewidth=1.5, linestyle="--", label="Rolling 28d")

ax.axvline(ex_val["day_idx"].min() - 0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_title(f"Series {example_sid3}: Forecast overlay (validation window)")
ax.set_xlabel("day_idx")
ax.set_ylabel("sale_amount")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 6c. D2 \u2014 Recovery First: Impute Censored Hours, Then Forecast

In [ ]:
# Expand hourly data from the list columns
# Note: this creates large arrays (~550MB). Colab free tier (12GB RAM) handles this fine.
print("Expanding hourly data...")

hourly_sales = np.stack(history["hours_sale"].values)          # (N, 24)
hourly_stock_ds = np.stack(history["hours_stock_status"].values)  # (N, 24)

# Focus on operating window h06..h21 (indices 6..21, 16 hours)
op_sales = hourly_sales[:, 6:22].astype(np.float32)
op_stock = hourly_stock_ds[:, 6:22].astype(np.float32)

# Mark censored hours
op_sales_masked = np.where(op_stock == 1, np.nan, op_sales)

total_cells = op_sales_masked.size
missing_cells = np.isnan(op_sales_masked).sum()
print(f"Operating window: {op_sales_masked.shape[1]} hours (h06-h21)")
print(f"Missing hourly cells: {missing_cells:,} / {total_cells:,} ({missing_cells/total_cells:.1%})")

In [ ]:
# --- Simple recovery: random pool sampling ---
visible_sum = np.nansum(np.where(op_stock == 0, op_sales, 0), axis=1)

imputed = op_sales_masked.copy()
imputed_count = 0
for h in range(16):
    col = imputed[:, h]
    mask = np.isnan(col)
    n_miss = mask.sum()
    if n_miss > 0:
        pool = col[~mask]
        imputed[mask, h] = np.maximum(0, rng.choice(pool, size=n_miss, replace=True))
        imputed_count += n_miss

# Rebuild corrected daily target
recovered_sum = np.nansum(imputed, axis=1)
outside_slice = np.maximum(history["sale_amount"].values.astype(np.float32) - visible_sum, 0)
recovered_daily = outside_slice + recovered_sum

history["recovered_daily_sales"] = recovered_daily

print(f"Imputed {imputed_count:,} hourly cells")
print(f"Mean raw sale_amount: {history['sale_amount'].mean():.4f}")
print(f"Mean recovered sales: {history['recovered_daily_sales'].mean():.4f}")

In [ ]:
# Re-split with recovered target
train_r, val_r = time_split(history, horizon=7)

# Seasonal naive on recovered target
val_start_r = val_r["day_idx"].min()
last_week_r = history[history["day_idx"].between(val_start_r - 7, val_start_r - 1)][
    ["series_id", "day_idx", "recovered_daily_sales"]
].copy()
last_week_r["forecast_day"] = last_week_r["day_idx"] + 7
last_week_r = last_week_r.rename(columns={"recovered_daily_sales": "pred_recovered_naive"})

val_r = val_r.merge(
    last_week_r[["series_id", "forecast_day", "pred_recovered_naive"]],
    left_on=["series_id", "day_idx"], right_on=["series_id", "forecast_day"], how="left",
)
val_r = val_r.drop(columns=["forecast_day"], errors="ignore")
fallback = train_r.groupby("series_id")["recovered_daily_sales"].mean()
val_r["pred_recovered_naive"] = val_r["pred_recovered_naive"].fillna(val_r["series_id"].map(fallback))

# Also add seasonal naive on raw for fair comparison
val_r = val_r.merge(
    val[["series_id", "day_idx", "pred_seasonal_naive"]].drop_duplicates(),
    on=["series_id", "day_idx"], how="left",
)

# Evaluate both
d2_results = {}
for method, col in [("Seasonal naive (raw)", "pred_seasonal_naive"), ("Seasonal naive (recovered)", "pred_recovered_naive")]:
    val_r["prediction"] = val_r[col].clip(lower=0)
    d2_results[method] = evaluate_forecast(val_r)

d2_df = pd.DataFrame(d2_results).T
print("=== D2 Recovery Comparison ===")
display(d2_df)

## 6f. D2 - 3 recovering strategy


In [8]:
# Sample of 5000 series
# Tạo một mẫu 5.000 series để sử dụng cho các tác vụ D2 (phục hồi)
# Điều này giúp giảm đáng kể mức sử dụng bộ nhớ và cho phép chạy trên Colab

num_series_to_sample = 5000

unique_series_ids = history['series_id'].unique()
if len(unique_series_ids) > num_series_to_sample:
    # Đảm bảo rng đã được khởi tạo (nếu chưa, hãy thêm dòng: rng = np.random.default_rng(RANDOM_SEED))
    sampled_series_ids = rng.choice(unique_series_ids, size=num_series_to_sample, replace=False)
    history_d2_sampled = history[history['series_id'].isin(sampled_series_ids)].copy()

    # Tùy chọn: Đặt lại `series_id` để chúng tuần tự từ 1 đến `num_series_to_sample`
    series_id_map_d2 = {old_id: new_id for new_id, old_id in enumerate(sampled_series_ids, 1)}
    history_d2_sampled['series_id'] = history_d2_sampled['series_id'].map(series_id_map_d2)

    print(f"\nĐã lấy mẫu {num_series_to_sample:,} series từ tổng số {len(unique_series_ids):,} series ban đầu cho tác vụ D2.")
    print(f"Hiện tại có {history_d2_sampled['series_id'].nunique():,} series trong dữ liệu D2 đã lấy mẫu.")
else:
    history_d2_sampled = history.copy() # Nếu số series ban đầu đã ít hơn, sử dụng toàn bộ dữ liệu
    print(f"\nSố lượng series ({len(unique_series_ids):,}) đã nhỏ hơn hoặc bằng {num_series_to_sample:,}. Không cần lấy mẫu cho D2.")

# Xác nhận kích thước mới và các series_id đã cập nhật
print(f"Tổng số hàng sau khi lấy mẫu cho D2: {len(history_d2_sampled):,}")
print(f"Các series_id duy nhất trong tập dữ liệu D2 đã lấy mẫu: {history_d2_sampled['series_id'].nunique()}")

# Cũng tạo val_d2_sampled từ val ban đầu để phù hợp với quy trình D2
# val_d2_sampled sẽ chỉ chứa các series_id có trong history_d2_sampled
val_d2_sampled = val[val['series_id'].isin(sampled_series_ids)].copy()
val_d2_sampled['series_id'] = val_d2_sampled['series_id'].map(series_id_map_d2)
print(f"Tổng số hàng trong tập validation D2 đã lấy mẫu: {len(val_d2_sampled):,}")


Đã lấy mẫu 5,000 series từ tổng số 50,000 series ban đầu cho tác vụ D2.
Hiện tại có 5,000 series trong dữ liệu D2 đã lấy mẫu.
Tổng số hàng sau khi lấy mẫu cho D2: 450,000
Các series_id duy nhất trong tập dữ liệu D2 đã lấy mẫu: 5000
Tổng số hàng trong tập validation D2 đã lấy mẫu: 35,000


In [19]:
# LGBMRegressor
from lightgbm import LGBMRegressor
# drop missing value
train_Lgbm_D2 = history_d2_sampled.dropna()
val_Lgbm_D2 = val_d2_sampled.dropna()

features = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount']

target = "sale_amount"

model = LGBMRegressor(random_state=1)
model.fit(train_Lgbm_D2[features], train_Lgbm_D2[target])

val_Lgbm_D2["prediction"] = model.predict(val_Lgbm_D2[features])
val_Lgbm_D2["prediction"] = val_Lgbm_D2["prediction"].clip(lower=0)

evaluate_forecast(val_Lgbm_D2)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023314 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1756
[LightGBM] [Info] Number of data points in the train set: 415000, number of used features: 11
[LightGBM] [Info] Start training from score 1.021034


{'wape_overall': np.float64(0.2455),
 'wape_low_sale': np.float64(0.3174),
 'wape_high_sale': np.float64(0.1925),
 'harmonic_mean': np.float64(0.2416),
 'scored_rows': 19818}

###  Per-Series Hourly Mean Recovered Data

In [9]:
print("--- Chuẩn bị dữ liệu giờ cho tập D2 đã lấy mẫu ---")
# Mở rộng dữ liệu giờ từ các cột list cho lịch sử đã lấy mẫu
hourly_sales_d2_sampled = np.stack(history_d2_sampled["hours_sale"].values)
hourly_stock_ds_d2_sampled = np.stack(history_d2_sampled["hours_stock_status"].values)

# Tập trung vào cửa sổ hoạt động h06..h21 (chỉ số 6..21, 16 giờ)
op_sales_d2_sampled = hourly_sales_d2_sampled[:, 6:22].astype(np.float32)
op_stock_d2_sampled = hourly_stock_ds_d2_sampled[:, 6:22].astype(np.float32)

# Đánh dấu các giờ bị censored
op_sales_masked_d2_sampled = np.where(op_stock_d2_sampled == 1, np.nan, op_sales_d2_sampled)

total_cells_d2 = op_sales_masked_d2_sampled.size
missing_cells_d2 = np.isnan(op_sales_masked_d2_sampled).sum()
print(f"Cửa sổ hoạt động (D2 đã lấy mẫu): {op_sales_masked_d2_sampled.shape[1]} giờ (h06-h21)")
print(f"Số ô bị thiếu hàng giờ (D2 đã lấy mẫu): {missing_cells_d2:,} / {total_cells_d2:,} ({missing_cells_d2/total_cells_d2:.1%})")



print("\n--- Triển khai phục hồi trung bình hàng giờ theo từng series (không rò rỉ dữ liệu) trên dữ liệu D2 đã lấy mẫu ---")

# Xác định ngày bắt đầu validation từ lịch sử D2 đã lấy mẫu cho dữ liệu huấn luyện
# Điều này đảm bảo rằng các giá trị trung bình chỉ được tính từ giai đoạn huấn luyện
val_start_pshm_calc_d2 = history_d2_sampled['day_idx'].max() - 7 + 1 # horizon là 7

# Lọc history_d2_sampled để tạo một tập chỉ huấn luyện cho việc tính toán giá trị trung bình
train_history_for_pshm_calc_d2 = history_d2_sampled[history_d2_sampled["day_idx"] < val_start_pshm_calc_d2].copy()

# Tính toán trung bình hàng giờ theo từng series CHỈ TỪ DỮ LIỆU HUẤN LUYỆN
per_series_hourly_mean_data_d2 = {}

train_series_ids_d2 = train_history_for_pshm_calc_d2['series_id'].unique()

for series_id in train_series_ids_d2:
    # Lấy các chỉ mục gốc của history_d2_sampled tương ứng với series_id này VÀ nằm trong tập huấn luyện
    original_index_labels_for_series_in_train_d2 = train_history_for_pshm_calc_d2[
        train_history_for_pshm_calc_d2['series_id'] == series_id
    ].index

    # Chuyển đổi các nhãn chỉ mục gốc này thành các chỉ mục vị trí 0 dựa trên
    # trong history_d2_sampled (và do đó là các mảng numpy op_sales_d2_sampled/op_stock_d2_sampled)
    positional_indices_for_series_in_train_d2 = history_d2_sampled.index.get_indexer(original_index_labels_for_series_in_train_d2)

    # Trích xuất dữ liệu bán hàng và trạng thái tồn kho của cửa sổ hoạt động cho series này
    # từ các mảng op_sales_d2_sampled/op_stock_d2_sampled ĐẦY ĐỦ, nhưng chỉ cho các hàng trong giai đoạn huấn luyện.
    series_op_sales_train_d2 = op_sales_d2_sampled[positional_indices_for_series_in_train_d2, :]
    series_op_stock_train_d2 = op_stock_d2_sampled[positional_indices_for_series_in_train_d2, :]

    # Che (mask) dữ liệu bán hàng cho các trường hợp hết hàng trong dữ liệu huấn luyện của series này
    series_op_sales_masked_train_d2 = np.where(series_op_stock_train_d2 == 1, np.nan, series_op_sales_train_d2)

    hourly_means_d2 = np.full(series_op_sales_masked_train_d2.shape[1], np.nan)

    for hour_idx in range(series_op_sales_masked_train_d2.shape[1]):
        hour_sales_data = series_op_sales_masked_train_d2[:, hour_idx]
        if not np.all(np.isnan(hour_sales_data)):
            hourly_means_d2[hour_idx] = np.nanmean(hour_sales_data)

    per_series_hourly_mean_data_d2[series_id] = hourly_means_d2

# Điền (impute) bằng trung bình hàng giờ theo từng series (áp dụng cho dữ liệu đã che đầy đủ bằng các giá trị trung bình từ huấn luyện)
imputed_pshm_d2 = op_sales_masked_d2_sampled.copy()

imputed_count_pshm_d2 = 0

for i in range(len(history_d2_sampled)): # Lặp qua từng quan sát series-ngày trong toàn bộ history_d2_sampled
    current_series_id = history_d2_sampled['series_id'].iloc[i]
    series_hourly_means = per_series_hourly_mean_data_d2.get(current_series_id)

    if series_hourly_means is not None: # Các giá trị trung bình tồn tại cho series này từ dữ liệu huấn luyện
        for h in range(16): # Lặp qua từng giờ trong cửa sổ hoạt động
            if np.isnan(imputed_pshm_d2[i, h]):
                mean_val = series_hourly_means[h]
                if not np.isnan(mean_val): # Chỉ điền nếu có giá trị trung bình cho series-giờ này
                    imputed_pshm_d2[i, h] = np.maximum(0, mean_val)
                    imputed_count_pshm_d2 += 1
                else:
                    # Dự phòng nếu giá trị trung bình của series-giờ cụ thể là NaN (ví dụ: series này luôn bị censored cho giờ này trong huấn luyện)
                    imputed_pshm_d2[i, h] = 0
    else:
        # Nếu một series_id nằm trong history_d2_sampled nhưng không nằm trong train_series_ids_d2 (ví dụ: nó chỉ xuất hiện trong validation),
        # hourly_means của nó sẽ không nằm trong per_series_hourly_mean_data_d2. Điền bằng 0 trong trường hợp này.
        for h in range(16):
            if np.isnan(imputed_pshm_d2[i, h]):
                imputed_pshm_d2[i, h] = 0


# Xây dựng lại target hàng ngày đã sửa (sử dụng imputed_pshm_d2 hiện bao gồm toàn bộ history_d2_sampled)
# visible_sum là tổng doanh số từ các giờ không bị censored (logic gốc)
visible_sum_pshm_d2 = np.nansum(np.where(op_stock_d2_sampled == 0, op_sales_d2_sampled, 0), axis=1)
recovered_sum_pshm_d2 = np.sum(imputed_pshm_d2, axis=1) # Tổng doanh số hàng giờ đã điền đầy đủ

# outside_slice là doanh số ngoài giờ hoạt động, được điều chỉnh để không âm
outside_slice_pshm_d2 = np.maximum(history_d2_sampled["sale_amount"].values.astype(np.float32) - visible_sum_pshm_d2, 0)
recovered_daily_pshm_d2 = outside_slice_pshm_d2 + recovered_sum_pshm_d2

history_d2_sampled["recovered_daily_sales_pshm"] = recovered_daily_pshm_d2

print(f"Đã điền {imputed_count_pshm_d2:,} ô hàng giờ bằng trung bình hàng giờ theo từng series (chỉ dữ liệu huấn luyện) trên dữ liệu D2 đã lấy mẫu.")
print(f"Doanh số trung bình thô (D2 đã lấy mẫu): {history_d2_sampled['sale_amount'].mean():.4f}")
print(f"Doanh số đã phục hồi trung bình (trung bình hàng giờ theo từng series, D2 đã lấy mẫu): {history_d2_sampled['recovered_daily_sales_pshm'].mean():.4f}")

# Chia lại với target đã phục hồi
train_r_pshm, val_r_pshm = time_split(history_d2_sampled, horizon=7)

# LGBMRegressor trên doanh số đã phục hồi trung bình hàng giờ theo từng series
from lightgbm import LGBMRegressor

# Drop các giá trị thiếu từ các dataframe train_r_pshm và val_r_pshm
train_Lgbm_pshm = train_r_pshm.dropna()
val_Lgbm_pshm = val_r_pshm.dropna()

features_lgbm = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount'
]

target_lgbm_pshm = "recovered_daily_sales_pshm"

model_pshm = LGBMRegressor(random_state=RANDOM_SEED)
model_pshm.fit(train_Lgbm_pshm[features_lgbm], train_Lgbm_pshm[target_lgbm_pshm])

val_Lgbm_pshm["prediction"] = model_pshm.predict(val_Lgbm_pshm[features_lgbm])
val_Lgbm_pshm["prediction"] = val_Lgbm_pshm["prediction"].clip(lower=0)

print("\n=== LGBMRegressor trên dữ liệu đã phục hồi bằng trung bình hàng giờ theo từng series (D2 đã lấy mẫu) ===")
val_Lgbm_pshm_eval = val_Lgbm_pshm.copy()
val_Lgbm_pshm_eval["sale_amount"] = val_Lgbm_pshm_eval[target_lgbm_pshm] # Để evaluate_forecast sử dụng target chính xác

pshm_lgbm_results = evaluate_forecast(val_Lgbm_pshm_eval)
print(pshm_lgbm_results)

# Kết hợp kết quả vào một DataFrame để so sánh
all_recovery_results_d2 = pd.DataFrame({
    "Per-Series Hourly Mean + LGBM (D2 Sampled)": pshm_lgbm_results
}).T
print("\n=== So sánh phục hồi D2 đã lấy mẫu ===")
display(all_recovery_results_d2)

--- Chuẩn bị dữ liệu giờ cho tập D2 đã lấy mẫu ---
Cửa sổ hoạt động (D2 đã lấy mẫu): 16 giờ (h06-h21)
Số ô bị thiếu hàng giờ (D2 đã lấy mẫu): 1,437,063 / 7,200,000 (20.0%)

--- Triển khai phục hồi trung bình hàng giờ theo từng series (không rò rỉ dữ liệu) trên dữ liệu D2 đã lấy mẫu ---
Đã điền 1,435,655 ô hàng giờ bằng trung bình hàng giờ theo từng series (chỉ dữ liệu huấn luyện) trên dữ liệu D2 đã lấy mẫu.
Doanh số trung bình thô (D2 đã lấy mẫu): 1.0039
Doanh số đã phục hồi trung bình (trung bình hàng giờ theo từng series, D2 đã lấy mẫu): 1.2258
Train: day 1..83 (415,000 rows), Val: day 84..90 (35,000 rows)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015384 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1753
[LightGBM] [Info] Number of data points in the train set: 380000, number of used features: 11
[LightGBM] [Info] Start traini

,wape_overall,wape_low_sale,wape_high_sale,harmonic_mean,scored_rows
Per-Series Hourly Mean + LGBM (D2 Sampled),0.258,0.3238,0.2095,0.2556,19818.0


### DLinear by Recency Recovered Data

In [13]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
import torch
from pypots.imputation import DLinear

# Install necessary libraries if not already present
!pip install -q pypots statsmodels



# --- Real DLinear Recovery using PyPOTS ---

print("\n--- Preparing data for PyPOTS (DLinear) ---")

# 1. Prepare 3D Tensors for PyPOTS
num_series_unique = history_d2_sampled['series_id'].nunique()
num_days_unique = history_d2_sampled['day_idx'].nunique()
num_hours_op = op_sales_masked_d2_sampled.shape[1] # 16 operating hours

# Create mappings for series_id and day_idx to 0-indexed integers
series_id_to_idx = {sid: i for i, sid in enumerate(sorted(history_d2_sampled['series_id'].unique()))}
day_idx_to_idx = {didx: i for i, didx in enumerate(sorted(history_d2_sampled['day_idx'].unique()))}

# Create empty 3D arrays to hold sales data for PyPOTS
X_sales_3d = np.full((num_series_unique, num_days_unique, num_hours_op), np.nan, dtype=np.float32)

# Populate X_sales_3d using history_d2_sampled and op_sales_masked_d2_sampled
# op_sales_masked_d2_sampled aligns with history_d2_sampled row-wise.
for original_df_idx, row_data in history_d2_sampled.iterrows():
    series_idx = series_id_to_idx[row_data['series_id']]
    day_idx_relative = day_idx_to_idx[row_data['day_idx']]

    # Get the integer position of the row in the current (filtered) history_d2_sampled DataFrame
    # This ensures correct indexing into op_sales_masked_d2_sampled which was stacked from `values`
    current_array_row_pos = history_d2_sampled.index.get_loc(original_df_idx)
    X_sales_3d[series_idx, day_idx_relative, :] = op_sales_masked_d2_sampled[current_array_row_pos, :]

# Create the mask M_sales_3d (1 for missing, 0 for observed)
M_sales_3d = np.isnan(X_sales_3d).astype(np.int32)

# Fill NaNs in X_sales_3d with 0, as PyPOTS expects numerical values where mask is 1
X_sales_3d[np.isnan(X_sales_3d)] = 0.0

# Convert to PyTorch tensors
X_tensor = torch.from_numpy(X_sales_3d)
M_tensor = torch.from_numpy(M_sales_3d)

# Determine the split point for training PyPOTS models (no future data leakage)
val_start_day_idx = val_d2_sampled['day_idx'].min() # This is the first day_idx in the validation set
train_days_count = day_idx_to_idx[val_start_day_idx] # Number of days in the training period (0-indexed length)

# Create a modified mask for training only: mask out all future days for training the imputation model
M_train_only = M_tensor.clone()
M_train_only[:, train_days_count:, :] = 1 # Set mask to 1 for all future days

train_data_for_pypots_fit = {'X': X_tensor, 'M': M_train_only} # Use full X, but with future masked in M for training


--- Preparing data for PyPOTS (DLinear) ---


In [14]:

# 2. DLinear Imputation with PyPOTS
print("\n--- DLinear Recovery (PyPOTS) ---")
# Ensure PyTorch device is set
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model_dlinear_pypots = DLinear(
    n_steps=num_days_unique, # Full sequence length for architecture
    n_features=num_hours_op,
    # individual_specific_layers=False, # Removed this argument as it causes TypeError
    moving_avg_window_size=25, # Added this argument as it was missing
    d_model=64, # Added this argument as it was missing
    epochs=5, # Reduced epochs for faster execution
    batch_size=32, # Batch size for training
    device=device
)

print("Training DLinear model...")
model_dlinear_pypots.fit(train_data_for_pypots_fit)

# Use the trained model to impute the entire dataset (full time span) with original missing mask
with torch.no_grad():
    imputed_X_dlinear_tensor = model_dlinear_pypots.impute({'X': X_tensor.clone(), 'M': M_tensor}) # Added .clone()
imputed_sales_dlinear_3d = imputed_X_dlinear_tensor

print("PyPOTS DLinear imputation complete. Now integrating and training LGBM.")

# 4. Integrate Imputed Data back to DataFrame structure
# Reshape imputed_sales_dlinear_3d back to (num_observations, num_hours)

imputed_op_sales_dlinear_pypots = np.zeros_like(op_sales_masked_d2_sampled)

# Map back from 3D to flat 2D using the original history_d2_sampled index order
for original_df_idx, row_data in history_d2_sampled.iterrows():
    series_idx = series_id_to_idx[row_data['series_id']]
    day_idx_relative = day_idx_to_idx[row_data['day_idx']]

    # Get the integer position of the row in the current (filtered) history_d2_sampled DataFrame
    # This ensures correct indexing into op_sales_masked_d2_sampled which was stacked from `values`
    current_array_row_pos = history_d2_sampled.index.get_loc(original_df_idx)
    imputed_op_sales_dlinear_pypots[current_array_row_pos, :] = imputed_sales_dlinear_3d[series_idx, day_idx_relative, :]


# Ensure non-negative sales
imputed_op_sales_dlinear_pypots = np.maximum(0, imputed_op_sales_dlinear_pypots)

# Calculate recovered daily sales for DLinear PyPOTS
visible_sum_dlinear_pypots = np.nansum(np.where(op_stock_d2_sampled == 0, op_sales_d2_sampled, 0), axis=1)
recovered_sum_dlinear_pypots = np.sum(imputed_op_sales_dlinear_pypots, axis=1)
outside_slice_dlinear_pypots = np.maximum(history_d2_sampled["sale_amount"].values.astype(np.float32) - visible_sum_dlinear_pypots, 0)
recovered_daily_dlinear_pypots = outside_slice_dlinear_pypots + recovered_sum_dlinear_pypots
history_d2_sampled["recovered_daily_sales_dlinear_pypots"] = recovered_daily_dlinear_pypots

2026-05-30 15:49:58 [INFO]: Using the given device: cuda
2026-05-30 15:49:58 [WARNING]: ‼️ saving_path not given. Model files and tensorboard file will not be saved.
2026-05-30 15:49:58 [INFO]: Using customized MAE as the training loss function.
2026-05-30 15:49:58 [INFO]: Using customized MSE as the validation metric function.
2026-05-30 15:49:58 [INFO]: DLinear initialized with the given hyperparameters, the number of trainable parameters: 22,684



--- DLinear Recovery (PyPOTS) ---
Using device: cuda
Training DLinear model...


2026-05-30 15:50:04 [INFO]: Epoch 001 - training loss (MAE): 0.1386
2026-05-30 15:50:07 [INFO]: Epoch 002 - training loss (MAE): 0.1160
2026-05-30 15:50:08 [INFO]: Epoch 003 - training loss (MAE): 0.1116
2026-05-30 15:50:10 [INFO]: Epoch 004 - training loss (MAE): 0.1095
2026-05-30 15:50:11 [INFO]: Epoch 005 - training loss (MAE): 0.1061
2026-05-30 15:50:11 [INFO]: Finished training. The best model is from epoch#5.


PyPOTS DLinear imputation complete. Now integrating and training LGBM.


In [15]:
# 5. Train LGBM and Evaluate
# Re-split data with new recovered targets
train_r_dlinear_pypots, val_r_dlinear_pypots = time_split(history_d2_sampled, horizon=7)

# Define features for LGBM (same as previous cells)
features_lgbm = [
    "sales_lag1", "sales_lag7", "sales_roll7", "sales_roll28", "avg_temperature",
    'management_group_id', 'city_id', 'product_id', 'holiday_flag', 'is_censored', 'discount'
]

# LGBMRegressor on DLinear PyPOTS recovered sales
train_Lgbm_dlinear_pypots = train_r_dlinear_pypots.dropna()
val_Lgbm_dlinear_pypots = val_r_dlinear_pypots.dropna()

target_lgbm_dlinear_pypots = "recovered_daily_sales_dlinear_pypots"
model_dlinear_lgbm_pypots = LGBMRegressor(random_state=RANDOM_SEED)
model_dlinear_lgbm_pypots.fit(train_Lgbm_dlinear_pypots[features_lgbm], train_Lgbm_dlinear_pypots[target_lgbm_dlinear_pypots])
val_Lgbm_dlinear_pypots["prediction"] = model_dlinear_lgbm_pypots.predict(val_Lgbm_dlinear_pypots[features_lgbm]).clip(min=0) # Changed lower=0 to min=0

print("\n=== LGBMRegressor on DLinear PyPOTS Recovered Data (D2 Sampled) ===")
val_Lgbm_dlinear_pypots_eval = val_Lgbm_dlinear_pypots.copy()
val_Lgbm_dlinear_pypots_eval["sale_amount"] = val_Lgbm_dlinear_pypots_eval[target_lgbm_dlinear_pypots]
dlinear_pypots_lgbm_results = evaluate_forecast(val_Lgbm_dlinear_pypots_eval)
print(dlinear_pypots_lgbm_results)

# --- Combine all results ---
# Re-initialize all_recovery_results_d2 to only include Per-Series Hourly Mean
# This ensures the comparison table is updated with the real models and not the simplified ones.
# Get existing pshm result (from kernel state if still available, or re-compute if needed).
# Assuming `all_recovery_results_d2` from previous cell still contains 'Per-Series Hourly Mean + LGBM (D2 Sampled)'
# and no other previous simplified results.

# This copies the first row (PSHM) from the existing DataFrame
if "all_recovery_results_d2" in locals() and not all_recovery_results_d2.empty:
    pshm_result_row = all_recovery_results_d2.loc[["Per-Series Hourly Mean + LGBM (D2 Sampled)"]]
    all_recovery_results_d2_updated = pd.DataFrame(pshm_result_row)
else:
    # Fallback if PSHM result is not easily accessible, though it should be.
    # In a real run, ensure pshm_lgbm_results is available or re-evaluate PSHM here.
    all_recovery_results_d2_updated = pd.DataFrame({
        "Per-Series Hourly Mean + LGBM (D2 Sampled)": pshm_lgbm_results # assuming pshm_lgbm_results is still in scope
    }).T


new_results_to_add_pypots = pd.DataFrame({
    "DLinear-PyPOTS + LGBM (D2 Sampled)": dlinear_pypots_lgbm_results
}).T

all_recovery_results_d2 = pd.concat([all_recovery_results_d2_updated, new_results_to_add_pypots])

print("\n=== So sánh tất cả các phương pháp phục hồi D2 đã lấy mẫu ===")
display(all_recovery_results_d2)

Train: day 1..83 (415,000 rows), Val: day 84..90 (35,000 rows)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012727 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1753
[LightGBM] [Info] Number of data points in the train set: 380000, number of used features: 11
[LightGBM] [Info] Start training from score 1.003605

=== LGBMRegressor on DLinear PyPOTS Recovered Data (D2 Sampled) ===
{'wape_overall': np.float64(0.2565), 'wape_low_sale': np.float64(0.3208), 'wape_high_sale': np.float64(0.2091), 'harmonic_mean': np.float64(0.2543), 'scored_rows': 19818}

=== So sánh tất cả các phương pháp phục hồi D2 đã lấy mẫu ===


,wape_overall,wape_low_sale,wape_high_sale,harmonic_mean,scored_rows
Per-Series Hourly Mean + LGBM (D2 Sampled),0.2580,0.3238,0.2095,0.2556,19818.0
DLinear-PyPOTS + LGBM (D2 Sampled),0.2565,0.3208,0.2091,0.2543,19818.0


### 6d. Error Analysis

In [ ]:
# WAPE by management group
scored = val[val["stock_hour6_22_cnt"] == 0].copy()
scored["prediction"] = scored["pred_seasonal_naive"].clip(lower=0)
scored["abs_error"] = np.abs(scored["sale_amount"] - scored["prediction"])

group_wape = scored.groupby("management_group_id").apply(
    lambda g: compute_wape(g["sale_amount"].values, g["prediction"].values),
    include_groups=False
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
group_wape.plot(kind="barh", color="#1C7293", edgecolor="white", ax=ax)
ax.set_title("WAPE by management group (seasonal naive baseline)")
ax.set_xlabel("WAPE")
ax.set_ylabel("Management Group ID")
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution and error vs stockout frequency
scored["residual"] = scored["sale_amount"] - scored["prediction"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(scored["residual"].clip(-5, 5), bins=60, color="#065A82", edgecolor="white")
axes[0].axvline(0, color="#E74C3C", linestyle="--", linewidth=2)
axes[0].set_title("Residual distribution (seasonal naive)")
axes[0].set_xlabel("Actual - Predicted")
axes[0].set_ylabel("Count")

series_error = scored.groupby("series_id").agg(
    mean_abs_error=("abs_error", "mean"),
).reset_index()
series_error = series_error.merge(
    history.groupby("series_id")["is_censored"].mean().rename("stockout_freq"),
    on="series_id"
)
axes[1].scatter(series_error["stockout_freq"], series_error["mean_abs_error"], alpha=0.1, s=5, color="#065A82")
axes[1].set_title("Mean absolute error vs stockout frequency")
axes[1].set_xlabel("Stockout frequency")
axes[1].set_ylabel("Mean absolute error")

plt.tight_layout()
plt.show()

---
---
## 7. Next Steps

### Operations Track

**O1 \u2014 Diagnosis First**
- Extend the heatmaps to find which store x category combinations are most fragile
- Test whether promotions (`discount < 1`) increase late-day stockouts
- Run panel regressions with fixed effects to isolate drivers

**O2 \u2014 Decision First**
- Build a simple corrected demand estimate (impute censored hours from `hours_sale`)
- Compute newsvendor order quantities under raw vs. corrected demand
- Visualize the service vs. waste trade-off curve

### Data Science Track

**D1 \u2014 Direct Benchmark**
- Try exponential smoothing or a simple LightGBM with lag features
- Analyze errors by day-of-week to detect weekly patterns
- Compare WAPE across cities to find geographic patterns

**D2 \u2014 Recovery First**
- Try per-series mean imputation instead of global pool sampling
- Compare multiple recovery strategies on the same baseline
- Focus error analysis on high-stockout series where recovery matters most

### Cross-Track Synergies
- Operations insights (which products are most fragile) can inform DS feature engineering
- DS demand recovery estimates can feed back into operations policy evaluation
- Both tracks benefit from understanding the hourly censoring structure

In [ ]:
print("Implementing Per-Series Hourly Mean recovery (leakage-free)...")

# --- Start: Necessary definitions moved from fyqpkO6BPLFc to ensure scope ---
# Expand hourly data from the list columns
# Note: this creates large arrays (~550MB). Colab free tier (12GB RAM) handles this fine.
print("Expanding hourly data...")

hourly_sales = np.stack(history["hours_sale"].values)          # (N, 24)
hourly_stock_ds = np.stack(history["hours_stock_status"].values)  # (N, 24)

# Focus on operating window h06..h21 (indices 6..21, 16 hours)
op_sales = hourly_sales[:, 6:22].astype(np.float32)
op_stock = hourly_stock_ds[:, 6:22].astype(np.float32)

# Mask censored hours
op_sales_masked = np.where(op_stock == 1, np.nan, op_sales)

total_cells = op_sales_masked.size
missing_cells = np.isnan(op_sales_masked).sum()
print(f"Operating window: {op_sales_masked.shape[1]} hours (h06-h21)")
print(f"Missing hourly cells: {missing_cells:,} / {total_cells:,} ({missing_cells/total_cells:.1%})")
# --- End: Necessary definitions moved ---


# Determine the validation start day from the original time_split for training data
# This ensures the means are calculated only from the training period
val_start_pshm_calc = history['day_idx'].max() - 7 + 1 # horizon is 7

# Filter history to create a training-only view for mean calculation
train_history_for_pshm_calc = history[history["day_idx"] < val_start_pshm_calc].copy()

# Calculate per-series hourly means FOR TRAINING DATA ONLY
per_series_hourly_mean_data = {}

# Iterate through unique series_id present in the training data
train_series_ids = train_history_for_pshm_calc['series_id'].unique()

for series_id in train_series_ids:
    # Get the original history indices that correspond to this series_id AND are in the training set
    original_history_indices_for_series_in_train = train_history_for_pshm_calc[train_history_for_pshm_calc['series_id'] == series_id].index

    # Extract the operating window sales and stock status for this series
    # from the FULL op_sales/op_stock arrays, but only for the training period rows.
    series_op_sales_train = op_sales[original_history_indices_for_series_in_train, :]
    series_op_stock_train = op_stock[original_history_indices_for_series_in_train, :]

    # Mask sales for stockouts within this series' training data
    series_op_sales_masked_train = np.where(series_op_stock_train == 1, np.nan, series_op_sales_train)

    # Initialize hourly_means with NaN for all hours
    hourly_means = np.full(series_op_sales_masked_train.shape[1], np.nan)

    # Calculate the mean for each hour for this specific series, ignoring NaNs
    # This mean is based ONLY on training data for this series, avoiding leakage.
    # Explicitly check for all-NaN slices to prevent RuntimeWarning
    for hour_idx in range(series_op_sales_masked_train.shape[1]):
        hour_sales_data = series_op_sales_masked_train[:, hour_idx]
        if not np.all(np.isnan(hour_sales_data)):
            hourly_means[hour_idx] = np.nanmean(hour_sales_data)

    per_series_hourly_mean_data[series_id] = hourly_means

# Impute using per-series hourly mean (apply to the full masked data using means from training)
imputed_pshm = op_sales_masked.copy() # Start with the full masked data

imputed_count_pshm = 0

for i in range(len(history)): # Iterate over each series-day observation in the full history
    current_series_id = history['series_id'].iloc[i]

    # Get the pre-calculated hourly means for this series (derived from training data)
    series_hourly_means = per_series_hourly_mean_data.get(current_series_id)

    if series_hourly_means is not None: # Means exist for this series from training data
        for h in range(16): # Iterate over each hour in the operating window
            if np.isnan(imputed_pshm[i, h]):
                mean_val = series_hourly_means[h]
                if not np.isnan(mean_val): # Only impute if a mean exists for this series-hour
                    imputed_pshm[i, h] = np.maximum(0, mean_val)
                    imputed_count_pshm += 1
                else:
                    # Fallback if specific series-hour mean is NaN (e.g., this series always censored for this hour in training)
                    imputed_pshm[i, h] = 0
    else:
        # If a series_id is in history but not in train_series_ids (e.g., it only appears in validation),
        # its hourly_means won't be in per_series_hourly_mean_data. Impute with 0 in this case.
        for h in range(16):
            if np.isnan(imputed_pshm[i, h]):
                imputed_pshm[i, h] = 0


# Rebuild corrected daily target (using the imputed_pshm which now covers the full history)
# visible_sum is from the original op_sales (non-censored hours)
visible_sum_pshm = np.nansum(np.where(op_stock == 0, op_sales, 0), axis=1)
recovered_sum_pshm = np.sum(imputed_pshm, axis=1) # Sum of the (now) fully imputed hourly sales

# outside_slice is sales outside operating hours, adjusted to be non-negative
outside_slice_pshm = np.maximum(history["sale_amount"].values.astype(np.float32) - visible_sum_pshm, 0)
recovered_daily_pshm = outside_slice_pshm + recovered_sum_pshm

history["recovered_daily_sales_pshm"] = recovered_daily_pshm

print(f"Imputed {imputed_count_pshm:,} hourly cells using per-series hourly mean (training data only).")
print(f"Mean raw sale_amount: {history['sale_amount'].mean():.4f}")
print(f"Mean recovered sales (per-series hourly mean): {history['recovered_daily_sales_pshm'].mean():.4f}")

# Re-split with recovered target (this will now use the newly updated history)
train_r_pshm, val_r_pshm = time_split(history, horizon=7)

# LGBMRegressor on per-series hourly mean recovered sales
from lightgbm import LGBMRegressor

# Drop missing values from the train_r_pshm and val_r_pshm dataframes
# (these were already created from history with 'recovered_daily_sales_pshm')
train_Lgbm_pshm = train_r_pshm.dropna()
val_Lgbm_pshm = val_r_pshm.dropna()

features_lgbm = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount'
]

target_lgbm_pshm = "recovered_daily_sales_pshm"

model_pshm = LGBMRegressor(random_state=1)
model_pshm.fit(train_Lgbm_pshm[features_lgbm], train_Lgbm_pshm[target_lgbm_pshm])

val_Lgbm_pshm["prediction"] = model_pshm.predict(val_Lgbm_pshm[features_lgbm])
val_Lgbm_pshm["prediction"] = val_Lgbm_pshm["prediction"].clip(lower=0)

print("=== LGBMRegressor on Per-Series Hourly Mean Recovered Data (Leakage-Free) ===")
# When evaluating, 'sale_amount' should be the actual target, which is recovered_daily_sales_pshm
val_Lgbm_pshm_eval = val_Lgbm_pshm.copy()
val_Lgbm_pshm_eval["sale_amount"] = val_Lgbm_pshm_eval[target_lgbm_pshm]

pshm_lgbm_results = evaluate_forecast(val_Lgbm_pshm_eval)
print(pshm_lgbm_results)

# If all_recovery_results is not in scope, this would need re-initialization or a new comparison table
# For now, let's just display it and assume it can be merged later.


Implementing Per-Series Hourly Mean recovery (leakage-free)...
Expanding hourly data...
Operating window: 16 hours (h06-h21)
Missing hourly cells: 14,311,536 / 72,000,000 (19.9%)
